# Reverse Diffusion — Sample Generation Pipeline

Generates synthetic time-series samples from a trained CSDI checkpoint and saves them to `data/generated/toy/` in the same CSV + stats-pickle format expected by `Toy_dataset_analysis.ipynb`.

**To use in the analysis notebook** update its path variables:
```python
gen_dir       = "../data/generated/toy/"
gen_stats_path = "../data/generated/toy/fake_stats_generated.pkl"
```

## 1. Imports

In [ ]:
import os
import sys
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import yaml
from copy import deepcopy

# Make project root importable from the notebooks/ folder
sys.path.insert(0, os.path.abspath(".."))

from src.models.model_core import CSDIModel
from src.utils.WIP_processes import Diffusion_Processes

## 2. Parameters

Edit the three variables below before running the rest of the notebook.

In [ ]:
# ── Checkpoint ────────────────────────────────────────────────────────────────
# Name of the .pt file inside  checkpoints/csdi/
CHECKPOINT_NAME = "final_ep-10_step-130_sde-ve_lr-1e-04_N-1000_notlinear_layers-4_nheads-8_20260319_134930.pt"

# ── Generation ────────────────────────────────────────────────────────────────
N_SAMPLES = 10          # number of synthetic time-series to generate

# ── Time horizon ──────────────────────────────────────────────────────────────
# Business-day date range used to label the generated rows.
# The range must cover at least seq_len (252) trading days.
START_DATE = "2020-01-02"
END_DATE   = "2021-12-31"

# ── Optional: override the number of reverse diffusion steps ──────────────────
# None  → use the value stored in the checkpoint config (process.N, typically 1000)
# int   → e.g. 200 for a faster (lower quality) run
NUM_REVERSE_STEPS = None

## 3. Load checkpoint and instantiate model

In [ ]:
CHECKPOINT_PATH = os.path.join("..", "checkpoints", "csdi", CHECKPOINT_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt   = torch.load(CHECKPOINT_PATH, map_location=device)
config = ckpt["config"]

target_dim = int(config["data"]["target_dim"])   # K = 5 features
seq_len    = int(config["train"]["seq_len"])      # L = 252 time steps

model = CSDIModel(target_dim=target_dim, config=config, device=device).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Checkpoint : {CHECKPOINT_NAME}")
print(f"target_dim : {target_dim}  |  seq_len : {seq_len}")
print(f"SDE type   : {config['process']['sde_type']}")

## 4. Instantiate Diffusion_Processes

In [ ]:
processes = Diffusion_Processes(config["process"])

num_reverse_steps = NUM_REVERSE_STEPS if NUM_REVERSE_STEPS is not None else processes.N
print(f"Diffusion_Processes ready — SDE: {processes.sde_type}, N: {processes.N}, model_steps: {processes.model_steps}")
print(f"Reverse steps to use: {num_reverse_steps}")

## 5. Generate samples

We generate unconditionally: `cond_mask = 0` everywhere so the model receives no context a

In [ ]:
# Unconditional generation: no observed context
observed_data = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)
cond_mask     = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)

# Normalised time positions in [0, 1], matching the training dataloader's index_norm mode
observed_tp = torch.linspace(0.0, 1.0, seq_len, device=device).unsqueeze(0).expand(N_SAMPLES, -1)

samples = processes.reverse_process(
    model          = model,
    shape          = (N_SAMPLES, target_dim, seq_len),
    observed_data  = observed_data,
    cond_mask      = cond_mask,
    observed_tp    = observed_tp,
    num_steps      = num_reverse_steps,
    probability_flow = False,
    device         = device,
)  # → (N_SAMPLES, K, L)

print(f"\nGenerated tensor shape : {samples.shape}")
print(f"Mean  : {samples.mean().item():.4f}")
print(f"Std   : {samples.std().item():.4f}")
print(f"Range : [{samples.min().item():.4f}, {samples.max().item():.4f}]")

## 6. Save results to `data/generated/toy/`

Each sample is saved as a separate CSV (`FAKE_XXXX_<start>_<end>_processed.csv`) with columns `Date, Open, High, Low, Close, Volume`, mirroring the format of `data/fake_individual_gbm/`.  
A companion `fake_stats_generated.pkl` stores per-ticker `{"mean": ..., "std": ...}`

In [ ]:
FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume"]
OUT_DIR      = os.path.join("..", "data", "generated", "toy")
os.makedirs(OUT_DIR, exist_ok=True)

# Build a business-day date index of exactly seq_len days
dates = pd.bdate_range(start=START_DATE, end=END_DATE)
if len(dates) < seq_len:
    raise ValueError(
        f"Date range {START_DATE} → {END_DATE} yields only {len(dates)} business days "
        f"but seq_len = {seq_len}. Please extend END_DATE."
    )
dates = dates[:seq_len]

samples_np = samples.detach().cpu().numpy()  # (N_SAMPLES, K, L)
stats_dict = {}

for i in range(N_SAMPLES):
    ticker = f"FAKE_{i + 1:04d}"
    series = samples_np[i]                          # (K, L)

    df = pd.DataFrame(series.T, columns=FEATURE_COLS)          # (L, K)
    df.insert(0, "Date", dates.strftime("%d/%m/%Y"))

    filename = f"{ticker}_{START_DATE}_{END_DATE}_processed.csv"
    df.to_csv(os.path.join(OUT_DIR, filename), index=False)

    # Per-feature mean and std of this generated trajectory
    stats_dict[ticker] = {
        "mean": series.mean(axis=1).astype(np.float32),         # (K,)
        "std":  series.std(axis=1).clip(min=1e-8).astype(np.float32),
    }

stats_path = os.path.join(OUT_DIR, "fake_stats_generated.pkl")
with open(stats_path, "wb") as f:
    pickle.dump(stats_dict, f)

print(f"Saved {N_SAMPLES} CSV files  →  {OUT_DIR}")
print(f"Saved stats pickle          →  {stats_path}")